In [ ]:
import pandas as pd
import os
import bed_reader
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report, 
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from scipy.stats import mannwhitneyu, chi2_contingency
from datetime import datetime
import json
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

# first - read in all of the results

In [ ]:
sbp_5 = pd.read_csv('5_age_groups.csv', sep = ',')
sbp_all = pd.read_csv('combined_SBP.csv', sep = ',')

In [ ]:
rsids_5 = list(sbp_5['rsID'][sbp_5['phenotype'] == 'angiotensin_receptor_blocker_1'])

In [ ]:
rsids_all = list(sbp_all['rsID'][sbp_all['phenotype'] == 'angiotensin_receptor_blocker'])

In [ ]:
rsids = rsids_5 + rsids_all

In [ ]:
#now save the list of rsids and eid to filter the bed file to load it easily:
pd.DataFrame(rsids).to_csv('snps_to_keep.txt', index=False, header=False)

In [ ]:
arb = pd.read_csv(
    'arb-therapy-data.tsv',
    sep = '\t'
)

### at this point the large plink file was filter with the plink_filter.slurm

In [ ]:
bed = bed_reader.open_bed('filtered_for_arb_response.bed') #this is only filtered for participants
val = bed.read()

In [ ]:
my_snps = pd.DataFrame(val, columns = bed.sid)

In [ ]:
my_snps['eid'] = (bed.iid).astype(int)

### Aggregate ARB response data:

In [ ]:
def parse_prescription_dates(row):

    val = row.get("prescriptions")
    if not val or pd.isna(val):
        return (pd.NaT, pd.NaT)

    try:
        dicts = json.loads(val)
    except:
        return (pd.NaT, pd.NaT)

    if not isinstance(dicts, list) or len(dicts) == 0:
        return (pd.NaT, pd.NaT)

    dates = []
    for dct in dicts:
        date_str = dct.get("date")
        if date_str:
            dt = pd.to_datetime(date_str, errors="coerce")
            if pd.notna(dt):
                dates.append(dt)

    if len(dates) == 0:
        return (pd.NaT, pd.NaT)

    start_date = min(dates)
    end_date = max(dates)
    return (start_date, end_date)


In [ ]:
def aggregate_arb_data(df):

    date_info = df.apply(parse_prescription_dates, axis=1)
 
    df["start_date"] = date_info.apply(lambda x: x[0])
    df["end_date"] = date_info.apply(lambda x: x[1])

    arb_mask = (df["group"] == "angiotensin-ii receptor antagonists")
    all_arb_drugs = df.loc[arb_mask, "drug_name"].dropna().unique()
    all_arb_drugs = sorted(all_arb_drugs)

    df_sorted = df.sort_values(["eid", "start_date", "end_date"])


    aggregated_rows = []

    print("dates extracted")

    for eid, sub in df_sorted.groupby("eid"):
        sub_arb = sub[sub["group"] == "angiotensin-ii receptor antagonists"]
        num_arb_therapies = len(sub_arb)

        num_diff_drug_classes = sub["group"].nunique(dropna=True)

        num_diff_arbs = sub_arb["drug_name"].nunique(dropna=True)

     
        arb_doses = {}
        for drug_name in all_arb_drugs:
        
            mask_dn = (sub["drug_name"] == drug_name) & arb_mask
         
            if mask_dn.any():
                mean_dose = sub.loc[mask_dn, "dose"].dropna().mean()
                arb_doses[drug_name] = mean_dose if not pd.isna(mean_dose) else 0.0
            else:
                arb_doses[drug_name] = 0.0

     
        dose_ever_increased = False
     
        sub_no_na = sub.dropna(subset=["dose"]).copy()
      
        dose_values = sub_no_na["dose"].values
     
        for i in range(len(dose_values) - 1):
            if dose_values[i+1] > dose_values[i]:
                dose_ever_increased = True
                break

        if len(sub_arb) > 0:
            longest_arb_duration = sub_arb["duration"].max()
        else:
            longest_arb_duration = 0.0

    
        idx_longest = sub["duration"].idxmax()
        group_longest = sub.loc[idx_longest, "group"] if pd.notna(idx_longest) else None
        arb_is_longest_therapy = (group_longest == "angiotensin-ii receptor antagonists")

  
        idx_last = sub["end_date"].idxmax()
        if pd.isna(idx_last):
           
            arb_is_last_therapy = False
        else:
            grp_last = sub.loc[idx_last, "group"]
            arb_is_last_therapy = (grp_last == "angiotensin-ii receptor antagonists")

        idx_first = sub["start_date"].idxmin()
        if pd.isna(idx_first):
            arb_is_first_therapy = False
        else:
            grp_first = sub.loc[idx_first, "group"]
            arb_is_first_therapy = (grp_first == "angiotensin-ii receptor antagonists")

      
        arb_ever_augmented = False
        if len(sub_arb) > 0:
           
            if (sub_arb["with_diuretic"].any() or sub_arb["with_calcium_channel_blocker"].any()):
                arb_ever_augmented = True

        changed_from_arb = False
        if len(sub_arb) > 0:
          
            earliest_arb = sub_arb["start_date"].min()
            others = sub[(sub["start_date"] > earliest_arb) & (sub["group"] != "angiotensin-ii receptor antagonists")]
            if len(others) > 0:
                changed_from_arb = True

        row_dict = {
            "eid": eid,
            "num_arb_therapies": num_arb_therapies,
            "num_diff_drug_classes": num_diff_drug_classes,
            "num_diff_arbs": num_diff_arbs,
            "dose_ever_increased": dose_ever_increased,
            "longest_arb_duration": longest_arb_duration,
            "arb_is_longest_therapy": arb_is_longest_therapy,
            "arb_is_last_therapy": arb_is_last_therapy,
            "arb_is_first_therapy": arb_is_first_therapy,
            "arb_ever_augmented": arb_ever_augmented,
            "changed_from_arb": changed_from_arb,
        }

   
        for drug_name in all_arb_drugs:
            colname = f"avg_dose_{drug_name}"
            row_dict[colname] = arb_doses[drug_name]

        aggregated_rows.append(row_dict)

    final_df = pd.DataFrame(aggregated_rows)

    return final_df

In [ ]:
aggregated = aggregate_arb_data(arb)

In [ ]:
#aggregated.to_csv('agg_arb.csv')

In [ ]:
aggregated = pd.read_csv('agg_arb.csv')

In [ ]:
aggregated = aggregated.drop('avg_dose_Sacubitril/Valsartan', axis = 1)

In [ ]:
# remove participants that actually do not have ARB prescriptions:
aggregated = aggregated[aggregated['num_arb_therapies'] != 0]

In [ ]:
aggregated = aggregated.merge(my_snps, on='eid')

In [ ]:
cols_to_drop = [
    'ARTN', 'SPRR2E', 'HOMER1', 'RAB24',
    'FOXL3', 'PAXIP1', 'ADGRA1', 'BORCS5', 'DMC1'
]

aggregated.drop(columns=cols_to_drop, inplace=True)

### read in the adjusted phenos:

In [ ]:
adj = pd.read_csv(
    'agg_arb_bwith_bp_snps_and_bp.csv',
    sep = '\t'
)

In [ ]:
adj_cols = ['eid',
        'num_arb_therapies_ADJ',
        'num_diff_drug_classes_ADJ',
        'num_diff_arbs_ADJ',
        'longest_arb_duration_ADJ',
        'avg_dose_Candesartan Cilexetil_ADJ',
        'avg_dose_Eprosartan_ADJ',
        'avg_dose_Irbesartan_ADJ',
        'avg_dose_Losartan Potassium_ADJ',
        'avg_dose_Olmesartan_ADJ',
        'avg_dose_Telmisartan_ADJ',
        'avg_dose_Valsartan_ADJ',
        'dose_ever_increased_ADJ',
        'arb_is_longest_therapy_ADJ',
        'arb_is_last_therapy_ADJ',
        'arb_is_first_therapy_ADJ',
        'arb_ever_augmented_ADJ',
        'changed_from_arb_ADJ'
]

In [ ]:
aggregated = aggregated.merge(adj[adj_cols], how = "left", on = "eid")

In [ ]:
aggregated[adj_cols+['rs12513069', 'SLC35F2', 'PKD1']].to_csv('for-glm-drug-response.tsv', sep = '\t')

# extract overall frequencies of tested variants:

In [ ]:
freqs = pd.read_csv(
    'HT_X_WES.afreq',
    sep = '\t'
)

In [ ]:
freqs

In [ ]:
variants = ['SLC35F2','PKD1','LDLR','rs12513069','rs11591147','APOB','rs118039278','rs56393506','rs11648003',
'rs8110479','rs72658867','rs5742911','rs12608822','rs62120566','rs11881756','rs7412']

In [ ]:
freqs[freqs['ID'].isin(variants)]